## Pandas analysis

This exercise consists in analyzing a dataset containg timing information from a series of Time-to-Digital-Converters (TDC) implemented in a couple of FPGAs. Each measurement (i.e. each row of the input file) consists of a flag that specifies the type of message ('HEAD', which in this case is always 1), two addresses of the TDC providing the signal ('FPGA' and 'TDC_CHANNEL'), and the timing information ('ORBIT_CNT', 'BX_COUNTER', and 'TDC_MEAS'). Each TDC count corresponds to 25/30 ns, whereas a unit of BX_COUNTER corresponds to 25 ns, and the ORBIT_CNT is increased every 'x' BX_COUNTER. This allows to store the time in a similar way to hours, minutes and seconds.

In [1]:
# If you didn't download it yet, please get the relevant file now!
!wget -O data_000637.txt "https://www.dropbox.com/scl/fi/pkkpoxlm7beasryexpdf8/data_000637.txt?rlkey=rkm2em1v57hewglzelmin21c9&dl=1"

/bin/bash: wget: command not found


In [21]:
import numpy as np
import pandas as pd

1\. Create a Pandas DataFrame reading N rows of the 'data_000637.txt' dataset. Choose N to be smaller than or equal to the maximum number of rows and larger that 10k.


In [3]:
file_name = "data/data_000637.txt"

df = pd.read_csv(file_name)

df

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
0,1,0,123,3869200167,2374,26
1,1,0,124,3869200167,2374,27
2,1,0,63,3869200167,2553,28
3,1,0,64,3869200167,2558,19
4,1,0,64,3869200167,2760,25
...,...,...,...,...,...,...
1310715,1,0,62,3869211171,762,14
1310716,1,1,4,3869211171,763,11
1310717,1,0,64,3869211171,764,0
1310718,1,0,139,3869211171,769,0


In [4]:
N = 100000

df.head(N)

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
0,1,0,123,3869200167,2374,26
1,1,0,124,3869200167,2374,27
2,1,0,63,3869200167,2553,28
3,1,0,64,3869200167,2558,19
4,1,0,64,3869200167,2760,25
...,...,...,...,...,...,...
99995,1,0,64,3869201161,2378,29
99996,1,0,70,3869201161,2472,26
99997,1,0,58,3869201161,2558,0
99998,1,0,57,3869201161,2561,23


2\. Find out the number of BX in a ORBIT (the value 'x').


In [5]:
df.groupby("ORBIT_CNT").agg(['min', 'max']) # Trying to understand when the orbit changes

HEAD     FPGA     TDC_CHANNEL      BX_COUNTER       TDC_MEAS    
            min max  min max         min  max        min   max      min max
ORBIT_CNT                                                                  
3869200167    1   1    0   1           5  139       2374  3187        0  28
3869200168    1   1    0   1           1  139          7  3538        0  29
3869200169    1   1    0   1           1  139          1  2766        0  29
3869200170    1   1    0   1           1  139        846  3377        0  29
3869200171    1   1    0   1           1  139        125  3542        0  29
...         ...  ..  ...  ..         ...  ...        ...   ...      ...  ..
3869211167    1   1    0   1           1  139         46  3553        0  29
3869211168    1   1    0   1           1  139        215  3556        0  29
3869211169    1   1    0   1           1  139         44  3498        0  29
3869211170    1   1    0   1           1  139          6  3527        0  29
3869211171    1   1    0   1           4  139          9   769        0  29

[11001 rows x 10 columns]

<span style="color: gray;"> We can see that not every possible value it's covered every orbit, but most likely all possible values are covered after certain number of orbits, so in order to find X we just need to find the max BX_counter overall. </span>

In [6]:
print("X =", df["BX_COUNTER"].max())

X = 3563


3\. Find out how much the data taking lasted. You can either make an estimate based on the fraction of the measurements (rows) you read, or perform this check precisely by reading out the whole dataset.

<span style="color: gray;">
    Each TDC count corresponds to 25/30 ns, whereas a unit of BX_COUNTER corresponds to 25 ns, and the ORBIT_CNT is increased every x=3563 BX_COUNTER.
    We just need to read the first and last record and convert the time units of the difference.
</span>

In [11]:
first_record = df.sort_values(["ORBIT_CNT", "BX_COUNTER"]).head(1)

last_record = df.sort_values(["ORBIT_CNT", "BX_COUNTER"]).tail(1) 

pd.concat([first_record, last_record]) #Show first and last rows to verify

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
0,1,0,123,3869200167,2374,26
1310718,1,0,139,3869211171,769,0


In [18]:
initial_time = ((first_record.ORBIT_CNT*3563 + first_record.BX_COUNTER)*25 + first_record.TDC_MEAS)*25/30
initial_time = initial_time.iloc[0] # in nanoseconds

final_time = ((last_record["ORBIT_CNT"]*3563 + last_record["BX_COUNTER"])*25 + last_record["TDC_MEAS"])*25/30
final_time = final_time.iloc[0] # in nanoseconds

print("Total duration in seconds:", (final_time - initial_time)*10**-9)

Total duration in seconds: 0.8167842908125


4\. Create a new column with the absolute time in ns (as a combination of the other three columns with timing information).

In [19]:
df


,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
0,1,0,123,3869200167,2374,26
1,1,0,124,3869200167,2374,27
2,1,0,63,3869200167,2553,28
3,1,0,64,3869200167,2558,19
4,1,0,64,3869200167,2760,25
...,...,...,...,...,...,...
1310715,1,0,62,3869211171,762,14
1310716,1,1,4,3869211171,763,11
1310717,1,0,64,3869211171,764,0
1310718,1,0,139,3869211171,769,0


In [20]:
df["ABS_TIME"] = ((df["ORBIT_CNT"]*3563 + df["BX_COUNTER"])*25 + df["TDC_MEAS"])*25/30

df

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS,ABS_TIME
0,1,0,123,3869200167,2374,26,2.872075e+14
1,1,0,124,3869200167,2374,27,2.872075e+14
2,1,0,63,3869200167,2553,28,2.872075e+14
3,1,0,64,3869200167,2558,19,2.872075e+14
4,1,0,64,3869200167,2760,25,2.872075e+14
...,...,...,...,...,...,...,...
1310715,1,0,62,3869211171,762,14,2.872083e+14
1310716,1,1,4,3869211171,763,11,2.872083e+14
1310717,1,0,64,3869211171,764,0,2.872083e+14
1310718,1,0,139,3869211171,769,0,2.872083e+14


5\. Replace the values (all 1) of the HEAD column randomly with 0 or 1.


In [25]:
df["HEAD"] = pd.Series(np.random.randint(0,2 , len(df)))

df

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS,ABS_TIME
0,1,0,123,3869200167,2374,26,2.872075e+14
1,0,0,124,3869200167,2374,27,2.872075e+14
2,1,0,63,3869200167,2553,28,2.872075e+14
3,0,0,64,3869200167,2558,19,2.872075e+14
4,1,0,64,3869200167,2760,25,2.872075e+14
...,...,...,...,...,...,...,...
1310715,0,0,62,3869211171,762,14,2.872083e+14
1310716,1,1,4,3869211171,763,11,2.872083e+14
1310717,0,0,64,3869211171,764,0,2.872083e+14
1310718,0,0,139,3869211171,769,0,2.872083e+14


6\. Create a new DataFrame that contains only the rows with HEAD=1.


7\. Make two occupancy plots (one for each FPGA), i.e. plot the number of counts per TDC channel

8\. Use the groupby method to find out the noisy channels, i.e. the TDC channels with most counts (say the top 3)

9\. Count the number of unique orbits. Count the number of unique orbits with at least one measurement from TDC_CHANNEL=139
